In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis

In [16]:
def compute_statistics(window):
    stats = {}
    for col in window.columns:
        if window[col].dtype in ['int64', 'float64']:  # Numeric columns
            stats[f'{col}_MinV'] = np.min(window[col], axis = 0)
            stats[f'{col}_MaxV'] = np.max(window[col], axis = 0)
            stats[f'{col}_std'] = np.std(window[col], axis = 0)
            stats[f'{col}_AM'] = np.mean(window[col], axis = 0)
            stats[f'{col}_AAM'] = np.mean(np.abs(window[col]), axis = 0)
            stats[f'{col}_GM'] = np.exp(np.mean(np.log(window[col] + 1e-9), axis = 0))
            stats[f'{col}_HM'] = len(window[col]) / np.sum(1.0 / (window[col] + 1e-9), axis=0)
            stats[f'{col}_Sum'] = np.sum(window[col], axis=0)
            stats[f'{col}_Q1'] = np.percentile(window[col], 25, axis=0)
            stats[f'{col}_Median'] = np.median(window[col], axis=0)
            stats[f'{col}_Q3'] = np.percentile(window[col], 75, axis=0)
            stats[f'{col}_Variance'] = np.var(window[col], axis=0)
            stats[f'{col}_Skewness'] = skew(window[col], axis=0)
            stats[f'{col}_Kurtosis'] = kurtosis(window[col], axis=0)
            stats[f'{col}_Zero_Crossings'] = np.sum(np.diff(np.sign(window[col]), axis=0) != 0, axis=0)
            stats[f'{col}_Mean_Energy'] = np.mean(window[col]**2, axis=0)
            stats[f'{col}_Mean_Curve_Length'] = np.mean(np.abs(np.diff(window[col], axis=0)), axis=0)
            stats[f'{col}_Mean_Teager_Energy'] = np.mean(window[col][1:-1]**2 - window[col][:-2] * window[col][2:], axis=0)

    return pd.Series(stats)


def sliding_window(df, window_size, step_size):
    aggregated_windows = []
    for start in range(0, len(df) - window_size + 1, step_size):
        end = start + window_size
        window = df.iloc[start:end]
        aggregated_window = compute_statistics(window)
        aggregated_windows.append(aggregated_window)
    return pd.DataFrame(aggregated_windows)

In [5]:
def remove_outliers_iqr(df):
    # Create a copy of the DataFrame to avoid modifying the original
    df_cleaned = df.copy()

    # Iterate over each column
    for column in df_cleaned.columns:
        # Check if the column is numeric
        if pd.api.types.is_numeric_dtype(df_cleaned[column]):
            # Calculate Q1 (25th percentile) and Q3 (75th percentile)
            Q1 = df_cleaned[column].quantile(0.25)
            Q3 = df_cleaned[column].quantile(0.75)
            
            # Calculate IQR (Interquartile Range)
            IQR = Q3 - Q1
            
            # Determine bounds for outliers
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            # Filter out rows with outliers
            df_cleaned = df_cleaned[(df_cleaned[column] >= lower_bound) & (df_cleaned[column] <= upper_bound)]
    
    return df_cleaned

In [3]:
df = pd.read_csv(r"video_CLNF\367_CLNF.csv")
display(df.head())

,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x_h1,y_h1,z_h1,Tx,Ty,Tz,Rx,Ry,Rz,Target
0,539.708,540.822,545.210,553.041,571.726,598.925,627.157,656.826,686.802,713.751,...,0.038569,0.474053,-0.879651,30.5069,-29.5368,517.648,0.148777,-0.201764,0.080146,1
1,537.843,539.471,544.706,554.230,575.186,603.914,632.853,663.251,693.936,721.402,...,0.031394,0.509867,-0.859680,31.5470,-29.9731,519.429,0.169917,-0.199755,0.060848,1
2,537.143,539.062,544.597,554.708,576.499,605.960,635.266,665.326,696.208,724.373,...,0.027374,0.519299,-0.854154,31.6734,-30.1217,518.989,0.177864,-0.189799,0.050668,1
3,536.830,538.777,544.479,555.234,577.522,607.189,636.903,666.737,698.149,727.127,...,0.006426,0.505616,-0.862735,31.4080,-30.1385,514.627,0.159880,-0.173943,0.041390,1
4,537.834,539.333,544.383,553.630,573.471,601.106,629.766,660.177,692.565,722.524,...,-0.003470,0.510209,-0.860043,30.7031,-30.2052,512.537,0.170386,-0.175499,0.061776,1


In [4]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47610 entries, 0 to 47609
Columns: 379 entries,  x0 to Target
dtypes: float64(372), int64(7)
memory usage: 137.7 MB
None


In [6]:
#df = remove_outliers_iqr(df)

In [7]:
y = df["Target"]
X = df.drop(columns = ["Target"])

In [12]:
print(X.shape)

(4053, 378)


In [9]:
window_size = 30
step_size = 25

stat_df = sliding_window(X, window_size, step_size)

In [17]:
display(stat_df.iloc[:,14:28])

,x1_MinV,x1_MaxV,x1_std,x1_AM,x1_AAM,x1_HM,x1_Sum,x1_Q1,x1_Median,x1_Q3,x1_Variance,x1_Zero_Crossings,x1_Mean_Energy,x1_Mean_Curve_Length
0,498.486,518.169,4.751200,511.017167,511.017167,510.972918,15330.515,507.97350,510.6650,513.17800,22.573904,0.0,261161.118533,1.160724
1,498.486,523.879,5.619000,510.601133,510.601133,510.539234,15318.034,507.08900,511.7810,512.12775,31.573163,0.0,260745.090525,2.473310
2,503.487,524.128,5.966630,512.893800,512.893800,512.824555,15386.814,508.03800,512.2360,518.17575,35.600674,0.0,263095.650752,2.919483
3,511.857,521.180,1.798045,512.705167,512.705167,512.698945,15381.155,512.01500,512.2075,512.55000,3.232967,0.0,262869.820894,0.413448
4,510.820,512.361,0.476144,511.832367,511.832367,511.831923,15354.971,511.65200,511.9790,512.20750,0.226714,0.0,261972.598281,0.091828
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,498.042,520.743,6.832669,507.304100,507.304100,507.212987,15219.123,501.93000,503.9290,511.72850,46.685368,0.0,257404.135244,2.352379
157,490.604,516.336,6.426312,506.953033,506.953033,506.870553,15208.591,502.21125,508.9810,512.09025,41.297490,0.0,257042.675496,2.794345
158,500.488,507.735,2.380458,504.058133,504.058133,504.046916,15121.744,502.56225,503.2395,507.11675,5.666579,0.0,254080.268358,0.315069
159,494.478,505.631,2.980180,498.289467,498.289467,498.271740,14948.684,496.21200,496.7305,500.36325,8.881475,0.0,248301.274066,0.671931


In [20]:
df = pd.read_csv(r"datasets\DAIC-Woz\train_split_Depression_AVEC2017.csv")
display(df)

,Participant_ID,PHQ8_Binary,PHQ8_Score,Gender,PHQ8_NoInterest,PHQ8_Depressed,PHQ8_Sleep,PHQ8_Tired,PHQ8_Appetite,PHQ8_Failure,PHQ8_Concentrating,PHQ8_Moving
0,303,0,0,0,0,0,0.0,0,0,0,0,0
1,304,0,6,0,0,1,1.0,2,2,0,0,0
2,305,0,7,1,0,1,1.0,2,2,1,0,0
3,310,0,4,1,1,1,0.0,0,0,1,1,0
4,312,0,2,1,0,0,1.0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
102,485,0,2,1,0,1,0.0,0,0,0,0,1
103,486,0,4,0,1,1,0.0,1,0,1,0,0
104,487,0,0,0,0,0,0.0,0,0,0,0,0
105,488,0,0,0,0,0,0.0,0,0,0,0,0


In [23]:
df["PHQ8_Binary"].value_counts()

PHQ8_Binary
0    23
1    12
Name: count, dtype: int64

In [22]:
df = pd.read_csv(r"datasets\DAIC-Woz\dev_split_Depression_AVEC2017.csv")
display(df)

,Participant_ID,PHQ8_Binary,PHQ8_Score,Gender,PHQ8_NoInterest,PHQ8_Depressed,PHQ8_Sleep,PHQ8_Tired,PHQ8_Appetite,PHQ8_Failure,PHQ8_Concentrating,PHQ8_Moving
0,302,0,4,1,1,1,0,1,0,1,0,0
1,307,0,4,0,0,1,0,1,0,2,0,0
2,331,0,8,1,1,1,1,1,1,1,1,1
3,335,1,12,0,1,1,3,2,3,1,1,0
4,346,1,23,0,2,3,3,3,3,3,3,3
5,367,1,19,1,3,3,2,2,2,3,3,1
6,377,1,16,0,2,2,1,2,3,3,2,1
7,381,1,16,1,2,3,3,3,1,3,0,1
8,382,0,0,1,0,0,0,0,0,0,0,0
9,388,1,17,1,1,2,2,2,3,3,2,2
